# Hierarchical mergers of binary black holes

All the relevant information for the project are to be found in the pdf document present in the repo.
Note that you are assigned to project 2 (as the title said).

## Datasets 

Datasets are stored on Google Drive (link and description in the pdf document)

### Contacts

* Giuliano Iorio <giuliano.iorio@unipd.it>


## Libraries 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np


## Data load

Here we load the different files containing all the data that we will use for this project. We will merge all data in to one large file adding flags to the nature of the system and to the metallicitty so we don't loose information. This will make it easier in order to make an exploratory analysis

In [ ]:
# uncomment and run this code block to read and combine all nth_generation.txt files into a single DataFrame

#def read_nth_generation(path):
#    # read and repair header (merge tokens that start with '(' into previous token)
#    with open(path, 'r') as f:
#        header = f.readline().strip()
#    tokens = header.split()
#    names = []
#    for tok in tokens:
#        if tok.startswith('(') and names:
#            names[-1] = names[-1] + ' ' + tok
#        else:
#            names.append(tok)
#    # read remaining rows using whitespace splitting and assign repaired names
#    df = pd.read_csv(path, sep='\s+', header=None, names=names, skiprows=1, comment='#')
#    return df
#
#base_path = Path('fastcluster_comp_physA')
#all_data = []
#for filepath in base_path.glob('**/*/Dyn/*/nth_generation.txt'):
#    sys_origin = filepath.parts[-4].split('_')[0]
#    mettalictty = float(filepath.parts[-2])
#    df = read_nth_generation(filepath)
#    df['sys'] = sys_origin
#    df['met'] = mettalictty
#    all_data.append(df)
#
#df_final = pd.concat(all_data, ignore_index=True, sort=False)

Now we drop the columns that have no useful information for our classification task

In [ ]:
# after running the above code block once, you can comment it out and read the combined CSV file directly in the future, uncomment the following lines to drop unnecessary columns,
#  convert identifier to category, and save the cleaned DataFrame to a new CSV file. You can also inspect the columns and data types before saving.:


#df_final.drop(columns=['c5:theta1', 'c6:theta2', 'c7:SMA(Rsun)', 'c8:ecc','c10:SMAfin(cm)', 'c11:eccfin',
#       'c12:tpeters/Myr', 'c14:vkick/kms', 'c18:flag1', 'c19:flag2', 'c20:flag3', 'c21:flagSN', 'c22:flag_exch',
#       'c23:flag_t3bb', 'c24:flag_evap','c26:ecc(10Hz)'], inplace=True)
#df_final['c0:identifier'] = df_final['c0:identifier'].astype('category')
#df_final.columns
#df_final.to_csv('combined_nth_generation.csv', index=False)
df_final = pd.read_csv('combined_nth_generation.csv')

Now we separate the data into three different dataframes according to the type of system they belong to, this will make it easier incase we want to analyze the data in a same group

In [ ]:
df_GC = df_final[df_final['sys'] == 'GC'] 
df_NSC = df_final[df_final['sys'] == 'NSC']
df_YSC = df_final[df_final['sys'] == 'YSC']

## Exploratory Data Analysis (EDA)

Now we will perform an exploration of the different parameters to detect some outliers and clean the datasets in order to make sure we have consistent measurements. 

In [ ]:
print(df_final.columns)
print(df_final.shape)

In [ ]:
#from ydata_profiling import ProfileReport

In [ ]:
#profile_GC = ProfileReport(df_GC, title="Profiling Report for GC", explorative=True)
#profile_GC.to_file("GC_report.html")
#profile_YSC = ProfileReport(df_YSC, title="Profiling Report for YSC", explorative=True)
#profile_YSC.to_file("YSC_report.html")
#profile_NSC = ProfileReport(df_NSC, title="Profiling Report for NSC", explorative=True)
#profile_NSC.to_file("NSC_report.html")
#profile_total = ProfileReport(df_final, title="Profiling Report for total dataset", explorative=True)
#profile_total.to_file("total_report.html")

After seeing the reports we notice that the values of the identifier column has values that are not unique so we will have to see if the rest of the columns has the same values and the entire row is duplicated or anything is else is happening

In [33]:
id = df_final['c0:identifier'].unique()
for i in id[:2]:
    subset = df_final[df_final['c0:identifier'] == i]
    if len(subset) > 1:
        for col in df_final.columns:
            if subset[col].nunique() > 1:
                print(f"Identifier {i} has {subset[col].nunique()} different values in column {col}")

Identifier 0 has 2 different values in column c1:M1/Msun
Identifier 0 has 2 different values in column c2:M2/Msun
Identifier 0 has 2 different values in column c3:chi1
Identifier 0 has 2 different values in column c4:chi2
Identifier 0 has 2 different values in column c9:(tDF+min(t12,t3bb))/Myr
Identifier 0 has 2 different values in column c13:(ngen (tDF+t3bb+tpeters))/Myr
Identifier 0 has 2 different values in column c15:mrem/Msun
Identifier 0 has 2 different values in column c16:arem
Identifier 0 has 2 different values in column c17:vesc/kms
Identifier 0 has 2 different values in column c25:Mtot/Msun
Identifier 0 has 2 different values in column met
Identifier 8 has 5 different values in column c1:M1/Msun
Identifier 8 has 5 different values in column c2:M2/Msun
Identifier 8 has 5 different values in column c3:chi1
Identifier 8 has 5 different values in column c4:chi2
Identifier 8 has 5 different values in column c9:(tDF+min(t12,t3bb))/Myr
Identifier 8 has 5 different values in column 

After executing the code we arrived to the conclusion that the rows with the same identifier are the same system after some number of generations, because we see that the only column that has the same value is the 'nGen' column

In [ ]:
from itertools import combinations
from scipy.stats._continuous_distns import alpha
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_joint_per_category(dfs, labels, x_col, y_col, hue_col, quantile=1):
    # 1. Get all unique values for the category across all datasets
    unique_cats = sorted(pd.concat([d[hue_col] for d in dfs]).unique())
    
    # 2. Global limits (90th percentile) for consistent scale across all plots
    all_x = pd.concat([d[x_col] for d in dfs])
    all_y = pd.concat([d[y_col] for d in dfs])
    x_lim = all_x.quantile(quantile)
    y_lim = all_y.quantile(quantile)
    x_min, y_min = all_x.min(), all_y.min()

    colors = ["#3498db", "#e74c3c", "#2ecc71"] # Blue, Red, Green
    markers = ['o', 's', '^']

    # 3. Create one figure for each category
    for cat in unique_cats:
        # Initialize the JointGrid for this specific category
        g = sns.JointGrid(height=6)
        
        for i, df in enumerate(dfs):
            # Filter the current dataset for the current category
            subset = df[df[hue_col] == cat]
            
            if subset.empty:
                continue
            
            # Sample for the scatter plot (Safety for millions of rows)
            
            
            # --- Center Plot ---
            sns.scatterplot(data=subset, x=x_col, y=y_col, ax=g.ax_joint,
                            color=colors[i], marker=markers[i], alpha=1/len(subset[hue_col]), 
                            s=15, label=labels[i])
            
            sns.kdeplot(data=subset, x=x_col, y=y_col, ax=g.ax_joint,
                        color=colors[i], levels=4, alpha=1, warn_singular=True)
            
            # --- Marginal Histograms ---
            sns.histplot(data=subset, x=x_col, ax=g.ax_marg_x, color=colors[i],
                         element="step", fill=True, linewidth=1.5, common_norm=False,
                         binrange=(x_min, x_lim), alpha=0.2)
            
            sns.histplot(data=subset, y=y_col, ax=g.ax_marg_y, color=colors[i],
                         element="step", fill=True, linewidth=1.5, common_norm=False,
                         binrange=(y_min, y_lim), alpha=0.2)

        # 4. Final Formatting for this category plot
        g.ax_joint.set_xlim(x_min, x_lim)
        g.ax_joint.set_ylim(y_min, y_lim)
        
        # Set Title and Legend
        plt.suptitle(f"Feature: {hue_col} | Value: {cat}", y=1.02, fontsize=14, fontweight='bold')
        g.ax_joint.legend(title="Datasets", loc='lower right')
        
        plt.savefig(f"images/joint_plot_{hue_col}_{cat}.png", bbox_inches='tight')

# Usage:
for combo in combinations(['c1:M1/Msun', 'c2:M2/Msun', 'c3:chi1', 'c4:chi2','c9:(tDF+min(t12,t3bb))/Myr', 'c13:(ngen (tDF+t3bb+tpeters))/Myr','c15:mrem/Msun', 'c16:arem', 'c17:vesc/kms', 'c25:Mtot/Msun'],2):
    try:
        plot_joint_per_category([df_GC, df_YSC, df_NSC], ['GC', 'YSC', 'NSC'], combo[0], combo[1], 'met',0.99)
    except Exception as e:
        print(f"Error plotting {combo[0]} vs {combo[1]}: {e}")
        continue

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_joint_threshold_split(dfs, labels, x_col, y_col, hue_col, threshold):
    # 1. Global limits (90th percentile) for consistent scale
    all_x = pd.concat([d[x_col] for d in dfs])
    all_y = pd.concat([d[y_col] for d in dfs])
    x_lim = all_x.quantile(0.99)
    y_lim = all_y.quantile(0.99)
    x_min, y_min = all_x.min(), all_y.min()

    colors = ["#3498db", "#e74c3c", "#2ecc71"] # Blue, Red, Green
    markers = ['o', 's', '^']
    
    # Define the two conditions
    conditions = [
        (lambda x: x <= threshold, f"<= {threshold}"),
        (lambda x: x > threshold, f"> {threshold}")
    ]

    # 2. Loop through the two threshold groups
    for check, cond_name in conditions:
        g = sns.JointGrid(height=6)
        
        for i, df in enumerate(dfs):
            # Apply the threshold filter to the current dataset
            subset = df[check(df[hue_col])]
            
            if subset.empty:
                continue
            
            # Sample for performance (3.2M rows safety)
            
            
            # --- Center Plot ---
            sns.scatterplot(data=subset, x=x_col, y=y_col, ax=g.ax_joint,
                            color=colors[i], marker=markers[i], alpha=0.2, 
                            s=15, label=labels[i])
            
            sns.kdeplot(data=subset, x=x_col, y=y_col, ax=g.ax_joint,
                        color=colors[i], levels=3, alpha=0.5, warn_singular=True)
            
            # --- Marginal Histograms ---
            sns.histplot(data=subset, x=x_col, ax=g.ax_marg_x, color=colors[i],
                         element="step", fill=False, linewidth=1.5, 
                         common_norm=False, binrange=(x_min, x_lim))
            
            sns.histplot(data=subset, y=y_col, ax=g.ax_marg_y, color=colors[i],
                         element="step", fill=False, linewidth=1.5, 
                         common_norm=False, binrange=(y_min, y_lim))

        # 3. Final Polish for each Threshold Plot
        g.ax_joint.set_xlim(x_min, x_lim)
        g.ax_joint.set_ylim(y_min, y_lim)
        
        plt.suptitle(f"Threshold Group: {hue_col} {cond_name}", y=1.02, 
                     fontsize=14, fontweight='bold')
        g.ax_joint.legend(title="Datasets", loc='upper right')
        
        plt.savefig(f"images/joint_threshold_{hue_col}_{cond_name}.png", bbox_inches='tight')

# Usage:
for combo in combinations(['c1:M1/Msun', 'c2:M2/Msun', 'c3:chi1', 'c4:chi2','c9:(tDF+min(t12,t3bb))/Myr', 'c13:(ngen (tDF+t3bb+tpeters))/Myr','c15:mrem/Msun', 'c16:arem', 'c17:vesc/kms', 'c25:Mtot/Msun'],2):
    try:
        plot_joint_threshold_split([df_GC, df_YSC, df_NSC], ['GC', 'YSC', 'NSC'], combo[0], combo[1], 'c27:Ngen', threshold=2)
    except Exception as e:
        print(f"Error plotting {combo[0]} vs {combo[1]}: {e}")
        continue


## Classification

## Results